# Week 2: Datasets and Gap Framing

**NWR Coverage Gap Research — Summer 2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/W2NJL/nwr-gap-research/blob/main/week2_datasets_and_gaps.ipynb)

---

A gap in NWR coverage only matters if people live in it — and those people's vulnerability depends on what hazards threaten them and how well-equipped they are to receive warnings by other means.

This week you will:
1. Build a complete picture of NWR's hazard scope (far beyond tornadoes)
2. Identify the datasets that capture population, demographics, and risk
3. Begin loading and profiling those datasets
4. Frame the research questions that will drive Weeks 4–7

---
## Part 1: Hazard Taxonomy

NWR covers a much broader hazard set than severe thunderstorms and tornadoes. This matters for the research because **different hazards concentrate in different geographies** — a gap area in the inland Northwest faces different risks than a gap area on the Gulf Coast.

### Weather hazards

| Category | Event Types |
|----------|-------------|
| Convective | Tornado warning/watch, Severe thunderstorm warning/watch |
| Flooding | Flash flood warning/watch/advisory, Flood warning/watch/statement, River flood |
| Winter | Winter storm warning/watch/advisory, Blizzard warning, Ice storm warning, Wind chill warning/advisory, Lake-effect snow |
| Tropical | Hurricane warning/watch, Tropical storm warning/watch, Storm surge warning/watch, Tropical depression advisory |
| Wind | High wind warning/advisory, Extreme wind warning |
| Heat/Cold | Excessive heat warning/watch/advisory, Heat advisory, Wind chill warning |
| Visibility | Dense fog advisory, Blowing dust advisory |
| Fire weather | Red flag warning, Fire weather watch |

### Non-weather natural hazards

| Category | Event Types |
|----------|-------------|
| Seismic/coastal | Tsunami warning/watch/advisory/information, Earthquake information |
| Volcanic | Volcanic ash advisory, Ashfall warning/advisory |
| Space weather | Geomagnetic storm watch (issued via NWS Space Weather Prediction Center) |

### Civil and human-caused emergencies

| Category | Event Types |
|----------|-------------|
| Law enforcement | AMBER alert (child abduction), Law enforcement warning |
| Infrastructure | 911 telephone outage alert, Civil emergency message |
| Industrial | Hazardous materials warning, Nuclear power plant warning/watch |
| National | Presidential alert (national emergency) |

### Implication for gap analysis

When you identify a gap city, the hazard context matters:
- A gap city in Tornado Alley with a high mobile home rate is extremely high-risk — mobile homes offer almost no protection and the residents are least likely to have alternative alert channels
- A gap city on the Oregon coast faces tsunami risk; NWR is often the primary warning mechanism in coastal areas with poor cell coverage
- A gap city in the high desert Southwest may face flash flood risk that's severely underestimated by outsiders

In Weeks 4–5, you'll layer hazard data on top of gap cities to produce a multi-risk score. Week 2 is about getting those hazard datasets in hand.

---
## Part 2: Dataset Inventory

Below are the key datasets for this research. For each, note the source, what it contains, and how it connects to gap analysis.

---

### 1. NWR Station Data (already have it)
**Source:** RadioLand / FCC / NOAA — `wx_stations.csv` in this repo  
**Use:** Primary dataset. Transmitter locations, power, antenna height, WFO assignments.

---

### 2. US Cities / Population Centers
**Source:** US Census Bureau + SimpleMaps  
**Direct URL (same as lab):**
```
https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/us_cities.csv
```
**Contains:** 4,834 US cities with population ≥ 5,000 (contiguous 48 states)  
**Columns:** `city`, `state_id` (2-letter abbrev), `state_name`, `county_name`, `lat`, `lng`, `population`  
**Note:** Coordinate columns are `lat`/`lng` — not `latitude`/`longitude`. There is no FIPS column; joins to county-level datasets (NRI, RUCC) should be done on `state_id` + `county_name` with string normalization, or via a separate county FIPS crosswalk.  
**Use:** The population centers we're checking coverage for.

**For finer-grained analysis**, consider Census block groups or tracts from the ACS (see below).

---

### 3. Census ACS 5-Year Estimates (American Community Survey)
**Source:** US Census Bureau — `api.census.gov`  
**API documentation:** [census.gov/developers](https://www.census.gov/developers/)  
**Key variables for vulnerability:**

| ACS Variable | What It Captures | Relevance |
|-------------|------------------|-----------|
| `B19013_001E` | Median household income | Lower income → less likely to own dedicated NWR receiver |
| `B25024_010E` + `B25024_011E` | Mobile home count | Mobile homes offer no shelter from tornadoes; residents most at risk |
| `B01001_020E`–`B01001_025E` (+ female equivalents) | Population 65+ | Elderly residents less likely to have smartphone alerts; hard-of-hearing population depends more on dedicated receivers |
| `B16004_*` | English language proficiency | NWR broadcasts in English; limited-English households may not act on alerts |
| `B08301_*` | Means of transportation to work | Proxy for car ownership; people in cars at the time of a warning depend on NWR or WEA (Wireless Emergency Alerts) |

**Access method:** You can query ACS via the Census API directly in Python using the `census` library or raw `requests` calls. A Census API key is free at [api.census.gov/data/key_signup.html](https://api.census.gov/data/key_signup.html).

---

### 4. CDC/ATSDR Social Vulnerability Index (SVI)
**Source:** CDC ATSDR  
**Direct download:** [atsdr.cdc.gov/placeandhealth/svi](https://www.atsdr.cdc.gov/placeandhealth/svi/data_documentation_download.html)  
**What it is:** A composite score (0–1) at the census tract level across four themes:
- Socioeconomic status (income, poverty, unemployment, no high school diploma)
- Household characteristics (age 65+, age 17 and under, disability, single-parent households)
- Racial and ethnic minority status
- Housing type and transportation (mobile homes, crowded housing, no vehicle, group quarters)

**Relevance:** A high SVI score = a population with fewer resources to prepare for, respond to, and recover from disasters. These are exactly the communities that NWR's over-the-air broadcast (requiring no internet, no cell service, no power grid) is designed to serve. A gap city with a high SVI score is a compound vulnerability.

**Format:** CSV download by state or national, at census tract level. Includes lat/lon centroids.

---

### 5. FEMA National Risk Index (NRI)
**Source:** FEMA  
**Direct download:** [hazards.fema.gov/nri/data-resources](https://hazards.fema.gov/nri/data-resources)  
**What it is:** County-level composite scores combining:
- Expected annual loss from 18 natural hazards (tornadoes, hurricanes, floods, earthquakes, tsunamis, wildfires, winter storms, etc.)
- Social vulnerability (derived from SVI)
- Community resilience

Outputs a **Risk Index** score and a **Expected Annual Loss** dollar figure per county, per hazard.

**Relevance:** Directly answers the question "which gap areas face the most serious multi-hazard risk?" You can join NRI county data to gap cities via the NRI's `STATE` and `COUNTY` name columns (matching against `state_name` and `county_name` in the cities dataset), or by constructing a FIPS code crosswalk.

**Format:** CSV, ~3,100 rows (one per county). ~130 columns — the key ones are `RISK_SCORE`, `RISK_RATNG`, and the per-hazard `{HAZARD}_EALR` columns (Expected Annual Loss Rate).

---

### 6. NOAA Storm Events Database
**Source:** NOAA National Centers for Environmental Information (NCEI)  
**Direct URL:** [www.spc.noaa.gov/wcm/data/1950-2024_actual_tornadoes.csv](https://www.spc.noaa.gov/wcm/data/1950-2024_actual_tornadoes.csv) (tornadoes only)  
**Broader database:** [ncdc.noaa.gov/stormevents/ftp.jsp](https://www.ncdc.noaa.gov/stormevents/ftp.jsp) (all event types)  
**What it is:** Every recorded significant weather event in the US since 1950, with location, magnitude, deaths, injuries, and property/crop damage.  
**Relevance:** Ground-truth for which hazards actually occur where — validates the FEMA NRI risk scores with historical incident data.

---

### 7. USDA Rural-Urban Continuum Codes (RUCC)
**Source:** USDA Economic Research Service  
**Download:** [ers.usda.gov/data-products/rural-urban-continuum-codes](https://www.ers.usda.gov/data-products/rural-urban-continuum-codes/)  
**What it is:** A 1–9 scale classifying every US county by degree of urbanization and adjacency to metro areas. Codes 4–9 are non-metro (rural).  
**Relevance:** Rural counties are both more likely to have NWR gaps (sparser transmitter network) and less likely to have reliable cell coverage for Wireless Emergency Alerts (WEA). A gap city in a rural county has fewer backup alert systems.

---

### 8. FCC Broadband Data (for WEA context)
**Source:** FCC Broadband Data Collection  
**Download:** [broadbandmap.fcc.gov](https://broadbandmap.fcc.gov/data-download)  
**Relevance:** Wireless Emergency Alerts (WEA) depend on cell coverage. In areas with poor or no cell coverage, NWR is the only redundant over-the-air alert system. FCC broadband data shows which census blocks have no LTE/5G coverage — these blocks represent areas where NWR gap = total alert gap.

---
## Part 3: Load and Profile the Key Datasets

For each dataset below, load it and answer the profiling questions. This is not about analysis yet — it's about understanding what you have before Week 4.

In [1]:
import pandas as pd
import numpy as np

WX_URL     = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/wx_stations.csv"
CITIES_URL = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/us_cities.csv"

wx = pd.read_csv(WX_URL)
wx = wx[wx['country'] == 'USA'].dropna(subset=['latitude', 'longitude']).copy()
wx['longitude'] = wx['longitude'] * -1
wx = wx.reset_index(drop=True)

# Note: cities coordinate columns are 'lat' and 'lng' (not 'latitude'/'longitude')
cities = pd.read_csv(CITIES_URL)

print(f"NWR stations: {len(wx)}")
print(f"Cities: {len(cities)}")
print(f"\ncities columns: {list(cities.columns)}")


NWR stations: 1029
Cities: 4834

cities columns: ['city', 'state_id', 'state_name', 'county_name', 'lat', 'lng', 'population']


### D1 — Load FEMA National Risk Index

Download the NRI county-level CSV from FEMA (link in the dataset inventory above). Load it into a DataFrame called `nri`.

Answer:
1. How many counties are in the dataset?
2. What columns capture overall risk? Print the top 10 highest-risk counties by `RISK_SCORE`.
3. Which hazard types have their own Expected Annual Loss columns? List them.
4. What is the FIPS code column called? (You'll need this to join to cities.)

In [17]:
# Download from FEMA and load — or upload the CSV to Colab manually
nri = pd.read_csv('/NRI_Table_Counties.csv')

print(len(nri))
print(f"\nri columns: {list(nri.columns)}")
print(nri.nlargest(10, 'RISK_SCORE')[['STATE', 'COUNTY', 'RISK_SCORE', 'RISK_RATNG']])
ealr_cols = [c for c in nri.columns if c.endswith('_EALR')]
print([c for c in ealr_cols])
print([c for c in nri.columns if 'FIPS' in c.upper()])

3232

ri columns: ['OID_', 'NRI_ID', 'STATE', 'STATEABBRV', 'STATEFIPS', 'COUNTY', 'COUNTYTYPE', 'COUNTYFIPS', 'STCOFIPS', 'POPULATION', 'BUILDVALUE', 'AGRIVALUE', 'AREA', 'RISK_VALUE', 'RISK_SCORE', 'RISK_RATNG', 'RISK_SPCTL', 'EAL_SCORE', 'EAL_RATNG', 'EAL_SPCTL', 'EAL_VALT', 'EAL_VALB', 'EAL_VALP', 'EAL_VALPE', 'EAL_VALA', 'ALR_VALB', 'ALR_VALP', 'ALR_VALA', 'ALR_NPCTL', 'ALR_VRA_NPCTL', 'SOVI_SCORE', 'SOVI_RATNG', 'SOVI_SPCTL', 'RESL_SCORE', 'RESL_RATNG', 'RESL_SPCTL', 'RESL_VALUE', 'CRF_VALUE', 'AVLN_EVNTS', 'AVLN_AFREQ', 'AVLN_EXP_AREA', 'AVLN_EXPB', 'AVLN_EXPP', 'AVLN_EXPPE', 'AVLN_EXPT', 'AVLN_HLRB', 'AVLN_HLRP', 'AVLN_HLRR', 'AVLN_EALB', 'AVLN_EALP', 'AVLN_EALPE', 'AVLN_EALT', 'AVLN_EALS', 'AVLN_EALR', 'AVLN_ALRB', 'AVLN_ALRP', 'AVLN_ALR_NPCTL', 'AVLN_RISKV', 'AVLN_RISKS', 'AVLN_RISKR', 'CFLD_EVNTS', 'CFLD_AFREQ', 'CFLD_EXP_AREA', 'CFLD_EXPB', 'CFLD_EXPP', 'CFLD_EXPPE', 'CFLD_EXPT', 'CFLD_HLRB', 'CFLD_HLRP', 'CFLD_HLRR', 'CFLD_EALB', 'CFLD_EALP', 'CFLD_EALPE', 'CFLD_EALT', 'CF

### D2 — Load CDC Social Vulnerability Index

Download the national SVI CSV at the census tract level. Load it into a DataFrame called `svi`.

Answer:
1. What is the overall vulnerability score column called? - RPL_THEMES is the percent rank summed from all 4 themes.
2. What are the four theme scores? - 'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4'
3. What geographic identifiers does it include? (State, county, tract FIPS?) - ['ST', 'STATE', 'STCNTY', 'COUNTY', 'FIPS', 'LOCATION']
4. What fraction of census tracts have an SVI score in the top quartile (most vulnerable)? - 0.25001188777936284

In [28]:
# Download from CDC ATSDR and load — or upload the CSV to Colab manually
svi = pd.read_csv('/SVI_2022_US.csv')

print(len(svi))
print(f"svi columns: {list(svi.columns)}")
display(svi['RPL_THEMES'])
theme_cols = [c for c in svi.columns if c.startswith('RPL_THEME') and c != 'RPL_THEMES']
print(theme_cols)
geo_cols = [c for c in svi.columns if c in ['ST','STATE','STCNTY','COUNTY','FIPS','LOCATION']]
print(geo_cols)

q75 = svi['RPL_THEMES'].quantile(0.75)
frac = (svi['RPL_THEMES'] >= q75).mean()
print(frac)


84120
svi columns: ['ST', 'STATE', 'ST_ABBR', 'STCNTY', 'COUNTY', 'FIPS', 'LOCATION', 'AREA_SQMI', 'E_TOTPOP', 'M_TOTPOP', 'E_HU', 'M_HU', 'E_HH', 'M_HH', 'E_POV150', 'M_POV150', 'E_UNEMP', 'M_UNEMP', 'E_HBURD', 'M_HBURD', 'E_NOHSDP', 'M_NOHSDP', 'E_UNINSUR', 'M_UNINSUR', 'E_AGE65', 'M_AGE65', 'E_AGE17', 'M_AGE17', 'E_DISABL', 'M_DISABL', 'E_SNGPNT', 'M_SNGPNT', 'E_LIMENG', 'M_LIMENG', 'E_MINRTY', 'M_MINRTY', 'E_MUNIT', 'M_MUNIT', 'E_MOBILE', 'M_MOBILE', 'E_CROWD', 'M_CROWD', 'E_NOVEH', 'M_NOVEH', 'E_GROUPQ', 'M_GROUPQ', 'EP_POV150', 'MP_POV150', 'EP_UNEMP', 'MP_UNEMP', 'EP_HBURD', 'MP_HBURD', 'EP_NOHSDP', 'MP_NOHSDP', 'EP_UNINSUR', 'MP_UNINSUR', 'EP_AGE65', 'MP_AGE65', 'EP_AGE17', 'MP_AGE17', 'EP_DISABL', 'MP_DISABL', 'EP_SNGPNT', 'MP_SNGPNT', 'EP_LIMENG', 'MP_LIMENG', 'EP_MINRTY', 'MP_MINRTY', 'EP_MUNIT', 'MP_MUNIT', 'EP_MOBILE', 'MP_MOBILE', 'EP_CROWD', 'MP_CROWD', 'EP_NOVEH', 'MP_NOVEH', 'EP_GROUPQ', 'MP_GROUPQ', 'EPL_POV150', 'EPL_UNEMP', 'EPL_HBURD', 'EPL_NOHSDP', 'EPL_UNINSUR', 

,RPL_THEMES
0,0.3635
1,0.4155
2,0.4843
3,0.2386
4,0.2059
...,...
84115,0.4548
84116,0.5458
84117,0.4005
84118,0.6864


['RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4']
['ST', 'STATE', 'STCNTY', 'COUNTY', 'FIPS', 'LOCATION']
0.25001188777936284


### D3 — Load NOAA Storm Events (All Hazards)

The NOAA Storm Events Database publishes annual CSV files at:

```
https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/
```

The 2024 file (most recent complete year) is:
```
StormEvents_details-ftp_v1.0_d2024_c20260421.csv.gz
```
pandas can read `.gz` files directly — no manual decompression needed.

**Key columns:** `EVENT_TYPE`, `STATE` (full state name), `DEATHS_DIRECT`, `DEATHS_INDIRECT`, `INJURIES_DIRECT`, `BEGIN_LAT`, `BEGIN_LON`

Answer:
1. How many events are in the 2024 file? - 69801
2. What are all the unique `EVENT_TYPE` values? How many distinct types are there? - 50
['Thunderstorm Wind' 'Excessive Heat' 'Heavy Snow' 'Heat' 'Hail'
 'High Wind' 'Funnel Cloud' 'Heavy Rain' 'Tornado' 'Drought'
 'Lake-Effect Snow' 'Dust Storm' 'Winter Weather'
 'Marine Thunderstorm Wind' 'Winter Storm' 'Flood' 'Flash Flood'
 'Tropical Storm' 'Waterspout' 'Avalanche' 'Astronomical Low Tide'
 'Strong Wind' 'Lightning' 'Coastal Flood' 'Blizzard'
 'Extreme Cold/Wind Chill' 'Wildfire' 'Debris Flow' 'Rip Current'
 'Dense Fog' 'High Surf' 'Marine High Wind' 'Cold/Wind Chill' 'Ice Storm'
 'Frost/Freeze' 'Sneakerwave' 'Freezing Fog' 'Sleet' 'Marine Hail'
 'Dust Devil' 'Storm Surge/Tide' 'Marine Tropical Storm'
 'Hurricane (Typhoon)' 'Marine Dense Fog' 'Marine Strong Wind'
 'Marine Hurricane/Typhoon' 'Tropical Depression' 'Seiche'
 'Lakeshore Flood' 'Marine Tropical Depression']

3. Which event types account for the most deaths (`DEATHS_DIRECT` + `DEATHS_INDIRECT`)? - Excessive Heat       253
Heat                 219
Flash Flood          117
Tropical Storm        78
Rip Current           65
Thunderstorm Wind     53
Tornado               53
Cold/Wind Chill       39
Flood                 35
Winter Weather        25
4. Which states (use the `STATE` column) had the most events? - TEXAS           5678
OKLAHOMA        3504
CALIFORNIA      2588
KANSAS          2572
ILLINOIS        2568
IOWA            2532
MISSOURI        2362
NEW YORK        2268
NEBRASKA        2040
PENNSYLVANIA    1981

In [35]:
STORM_URL = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2024_c20260421.csv.gz"

# pandas reads .gz directly — this file is ~10MB and may take 15–30 seconds in Colab
storms = pd.read_csv(STORM_URL)

print(len(storms))
print(f"storms columns: {list(storms.columns)}")
print(storms['EVENT_TYPE'].nunique())
print(storms['EVENT_TYPE'].unique())
storms['DEATHS_TOTAL'] = storms['DEATHS_DIRECT'] + storms['DEATHS_INDIRECT']
print(storms.groupby('EVENT_TYPE')['DEATHS_TOTAL'].sum().sort_values(ascending=False).head(10))
print(storms['STATE'].value_counts().head(10))

69801
storms columns: ['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH', 'END_DAY', 'END_TIME', 'EPISODE_ID', 'EVENT_ID', 'STATE', 'STATE_FIPS', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'SOURCE', 'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY', 'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH', 'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON', 'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE']
50
['Thunderstorm Wind' 'Excessive Heat' 'Heavy Snow' 'Heat' 'Hail'
 'High Wind' 'Funnel Cloud' 'Heavy Rain' 'Tornado' 'Drought'
 'Lake-Effect Snow' 'Dust Storm' 'Winter Weather'
 'Marine Thunderstorm Wind' 'Winter S

### D4 — USDA Rural-Urban Continuum Codes

Download the RUCC Excel file from USDA ERS (link in dataset inventory). The sheet you want is `Rural-urban Continuum Code 2023`. Key columns are `FIPS`, `State` (2-letter), `County_Name`, and `RUCC_2023`.

**Joining to cities:** `cities` has no FIPS column — join on `State` + `County_Name` using `state_id` and `county_name`. Note that county name strings may not match exactly (e.g., "St. Louis" vs "Saint Louis") — you may need `.str.lower().str.strip()` normalization on both sides.

Answer:
1. How many cities in our dataset are in non-metro counties (RUCC ≥ 4)?
2. Of those, how many are in the most rural counties (RUCC ≥ 7)?
3. Do rural cities appear more or less frequently in states you'd expect to have NWR gaps?

In [37]:
# Download from USDA ERS (Excel or CSV format)
rucc = pd.read_csv('/Ruralurbancontinuumcodes2023.csv')

# YOUR CODE HERE


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in position 89596: invalid continuation byte

---
## Part 4: Research Questions

These are the questions that will drive the analysis in Weeks 4–7. Read through them now. Some you can already sketch an answer to based on what you've seen in Week 1 and the dataset profiles above.

**Coverage questions:**
1. What percentage of US cities with population ≥ 5,000 fall outside a 40-mile radius of any NWR transmitter?
2. How does that percentage change if we use a power-weighted radius (e.g., a 1,000-watt station gets 40 miles, a 100-watt station gets 20 miles, a 5-watt station gets 5 miles)?
3. Which states have the most gap cities in absolute terms? In percentage terms?

**Vulnerability questions:**
4. Do gap cities have higher average SVI scores than covered cities?
5. Do gap cities have higher rates of mobile home occupancy?
6. Do gap cities have higher proportions of elderly residents (65+)?
7. Do gap cities have lower English language proficiency?

**Hazard overlap questions:**
8. What is the average FEMA NRI Risk Score for gap cities vs. covered cities?
9. Do gap cities in Tornado Alley overlap with high mobile home rates? (Compound vulnerability)
10. How many gap cities are in counties with no LTE coverage (total alert gap)?
11. Which individual hazard types (from the NRI) are most overrepresented in gap counties?

**Network analysis questions:**
12. What is the optimal location for a new NWR transmitter to reduce the most population exposure?
13. Are there clusters of gap cities that a single new transmitter could cover simultaneously?
14. Which WFOs oversee the most gap cities within their service area — and are those areas also low-power?

**The top 50 list (Week 6–7):**
15. Combining coverage gap distance, SVI score, NRI risk, and rural isolation — what is a composite at-risk score for each gap city?
16. Which 50 gap cities score highest on that composite measure?

---
## Week 2 Reflection

Write brief answers to the following. These will become the methods section of your eventual FEMA report.

1. Of the datasets listed, which two do you think will be most important for identifying the *most at-risk* gap areas? Why?
2. The class lab used a flat 40-mile coverage radius for all stations. Name two ways this oversimplifies reality — and what data you now have access to that could improve the model.
3. A FEMA reviewer reading this research will ask: "Why should we prioritize NWR over cell-based Wireless Emergency Alerts?" Write two or three sentences answering that question using what you've learned.
4. Which hazard type, outside of tornadoes, do you think is most underappreciated in the NWR gap conversation, and why?

**Your answers:**

1.

2.

3.

4.
